<a href="https://colab.research.google.com/github/And123-web/FinalYearProject/blob/main/5_02_practice_weekly_ops.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=17M9oGaOj4KdKiDDH3oTibxJ9qBTSjp6n)

# 5.02 Analytics Engineering: practice test
## Practice: Weekly operations pipeline

## How this practice works

This is a rehearsal for the in-class test. It is **not marked** and nothing is submitted. It works only if you treat it like the real thing: attempt it first, and only then look at the model answer.

This is set to be the same time given in the real test; the same style of notebook and problem; and as close to the real thing as possible. However, of course, the problem is different; the data is different; and you will do it asychronously (which is also different).

- Set a timer for **55 minutes** and work alone. Stop when it rings, then spend **5 minutes** exporting your notebook (see below). Write your start and stop times in the cell under the timing table.
- Work in **Google Colab** and save the notebook as `.ipynb`.
- You may use *any materials, the module GitHub code and any generative AI tool*.
- There is no end-point as such. The task is longer than 55 minutes, so you are completing part of a larger job. Do as much as you can, and do it properly.
- The code itself is not marked. Marks in the real test go on how the task was executed, the methodology applied, and the interpretation of results. Practise that.
- In text cells or comments, include:
  1. **how the code was generated**: the prompt you sent, or the source you used;
  2. **evidence of checking**: what you compared, counted or inspected, and what it showed;
  3. **your interpretation in your own words**.

**Tips**

- Run small pieces and know what output you expect before you run them.
- Put enough text next to the code that someone else can follow your reasoning.
- Follow sound analytics engineering method: understand the grain/rows that you are building around, clean in stages, check before you move on.
- Quality over quantity. A smaller pipeline that you have checked beats a larger one you have not.
- Stay calm. Not finishing is expected.

**Suggested timing**

| Minutes | Focus |
| --- | --- |
| 0 to 10 | Explore and profile the tables |
| 10 to 20 | Plan: grain, joins, cleaning decisions |
| 20 to 45 | Build as far as you can |
| 45 to 55 | Check, reconcile and write up |
| 55 to 60 | Export: **Runtime, then Restart session and run all**; File, Download, .ipynb; put the file in a zip |

In the real test you upload the zip to the Class Test area on my.wbs. Here, keep it so you can compare your work with the model answer afterwards.



```
# This is formatted as code
```

**Started at:** ____  **Stopped at:** ____  **Furthest step I reached (see the brief):** ____

## The data

You have six tables from a fictional food-delivery marketplace, Plateful, which works with restaurants in five UK cities. They cover orders placed from 1 March to 30 June 2026.

| Name in this notebook | What it holds |
| --- | --- |
| `orders` | One record per order: customer, restaurant, when it was placed, delivery fee |
| `order_status_events` | The order's progress through the process, recorded as events |
| `order_lines` | The items inside each order |
| `payments` | Money movements for each order: charges and refunds |
| `restaurants` | Restaurant details: cuisine and city |
| `customers` | Customer details |

All timestamps in all tables are stored in **UTC**.

**How the tables connect**

- `orders` to `order_status_events`, `order_lines` and `payments` on `order_id`
- `orders` to `restaurants` on `restaurant_id`
- `orders` to `customers` on `customer_id`

Check the actual column names in the next cell before you write any joins.

In [1]:
# ---- Setup: run this cell first ----
!pip install -q duckdb

import glob
import os

import duckdb
import pandas as pd

DATA_URL = "https://github.com/MJMortensonWarwick/adminaeaiv/releases/download/practicesession/plateful-practice.zip"    # link to plateful-practice.zip (the CSV files are inside a folder in the zip)

DATA_DIR = "/content/plateful"

if DATA_URL and not glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True):
    import urllib.request
    import zipfile
    os.makedirs(DATA_DIR, exist_ok=True)
    urllib.request.urlretrieve(DATA_URL, "/content/plateful.zip")
    zipfile.ZipFile("/content/plateful.zip").extractall(DATA_DIR)

FILES = {
    "orders": "orders.csv",
    "order_status_events": "order_status_events.csv",
    "order_lines": "order_lines.csv",
    "payments": "payments.csv",
    "restaurants": "restaurants.csv",
    "customers": "customers.csv",
}

tables, missing = {}, []
for name, filename in FILES.items():
    found = glob.glob(os.path.join(DATA_DIR, "**", filename), recursive=True)
    if found:
        tables[name] = pd.read_csv(found[0])
    else:
        missing.append(filename)

if missing:
    found_csvs = sorted(os.path.basename(f) for f in glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True))
    raise FileNotFoundError(f"Missing files: {missing}. CSV files found under {DATA_DIR}: {found_csvs}.")

# Make every table available as a DuckDB table, so you can use SQL or pandas.
con = duckdb.connect()
for name, df in tables.items():
    con.register(name, df)


def q(sql):
    """Run SQL against the tables above and return a pandas DataFrame."""
    return con.sql(sql).df()


for name, df in tables.items():
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")
    print("   ", list(df.columns))

orders: 14,494 rows, 5 columns
    ['order_id', 'customer_id', 'restaurant_id', 'order_ts', 'delivery_fee']
order_status_events: 56,812 rows, 4 columns
    ['event_id', 'order_id', 'status', 'event_ts']
order_lines: 36,120 rows, 5 columns
    ['order_id', 'line_no', 'item_name', 'quantity', 'unit_price']
payments: 16,423 rows, 5 columns
    ['payment_id', 'order_id', 'payment_type', 'amount', 'paid_at']
restaurants: 67 rows, 5 columns
    ['restaurant_id', 'restaurant_name', 'cuisine', 'city', 'effective_from']
customers: 4,200 rows, 3 columns
    ['customer_id', 'signup_date', 'home_city']


## Examples: two ways to work with the tables

You can use SQL, pandas, or a mix of both. The next two cells show each, using only the tables and keys described above. Run them and change them to see what they do. You do not have to use them in your answer.

In [2]:
# Example 1: SQL
# q() runs SQL against any of the tables above and returns a pandas DataFrame.
from IPython.display import display

# Look at the first few rows of a table
display(q("SELECT * FROM orders LIMIT 5"))

# How many rows are there, and how many different order_id values?
display(q("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT order_id) AS n_distinct_orders FROM orders"))

# Join two tables on a key
display(q("""
    SELECT o.order_id, r.restaurant_id
    FROM orders AS o
    JOIN restaurants AS r ON o.restaurant_id = r.restaurant_id
    LIMIT 5
"""))

,order_id,customer_id,restaurant_id,order_ts,delivery_fee
0,O000001,C01161,R052,2026-03-01T09:13:23Z,1.49
1,O000002,C00723,R027,2026-03-01T09:15:33Z,2.49
2,O000003,C00781,R044,2026-03-01T09:23:42Z,0.00
3,O000004,C01430,R049,2026-03-01T09:28:14Z,1.49
4,O000005,C00039,R009,2026-03-01T11:32:35Z,0.00


,n_rows,n_distinct_orders
0,14494,14494


,order_id,restaurant_id
0,O000001,R052
1,O000002,R027
2,O000003,R044
3,O000004,R049
4,O000005,R009


In [3]:
# Example 2: pandas
# Each table is also a pandas DataFrame, stored in the dictionary `tables`.
from IPython.display import display

orders = tables["orders"]
display(orders.head())        # first few rows
print(orders.shape)           # (number of rows, number of columns)
print(orders.dtypes)          # data type of each column

# The same join as in Example 1, using pandas
joined = tables["orders"].merge(tables["restaurants"], on="restaurant_id", how="inner")
display(joined[["order_id", "restaurant_id"]].head())

# You can mix the two: run SQL, then use pandas methods on the result
result = q("SELECT customer_id FROM orders")
print(result["customer_id"].nunique(), "different customer_id values in orders")

,order_id,customer_id,restaurant_id,order_ts,delivery_fee
0,O000001,C01161,R052,2026-03-01T09:13:23Z,1.49
1,O000002,C00723,R027,2026-03-01T09:15:33Z,2.49
2,O000003,C00781,R044,2026-03-01T09:23:42Z,0.00
3,O000004,C01430,R049,2026-03-01T09:28:14Z,1.49
4,O000005,C00039,R009,2026-03-01T11:32:35Z,0.00


(14494, 5)
order_id          object
customer_id       object
restaurant_id     object
order_ts          object
delivery_fee     float64
dtype: object


,order_id,restaurant_id
0,O000001,R052
1,O000002,R027
2,O000003,R044
3,O000004,R049
4,O000005,R009


4073 different customer_id values in orders


## Your brief

**From:** Operations

> We run Plateful, a food-delivery marketplace in five UK cities. We are starting a weekly operations report and need the first layers of a data pipeline behind it: clean tables first, then one reporting table. For each week, show orders, net revenue and average delivery time by cuisine and by city. We also need to know how far we can trust the figures.

**What your final table should look like**

One table with columns for week, cuisine, city, number of orders, net revenue and average delivery time in minutes. State clearly what one row of your table represents.

**Decisions you must make and state**

- Which week does an order belong to?
- Which orders count as orders, and which do not?
- What does "net revenue" mean here, and where does it come from?
- What counts as "delivery time", and for which orders can it be measured?

**A suggested route, in order** (you are not expected to finish)

1. Profile the tables: grain, keys, duplicates, odd values.
2. Build clean tables (a silver layer) with one row per order where that is the grain.
3. Build the one reporting table (gold) from them.
4. Show with checks that the gold table agrees with the source data.
5. If you have time: add cancelled orders and a refund rate to the table, and write automated checks for your clean tables.

## 1. Understand the data

*Look before you build. Write down what you find: what one row of each table represents, how the tables connect, and anything that looks odd or missing.*

### 1. Data Profiling Findings

- **Orders**: 14,494 unique orders. Primary key is `order_id`.
- **Customers**: 4,200 unique customers.
- **Restaurants**: Contains Slowly Changing Dimension (SCD Type 2) records. 3 restaurants (`R007`, `R021`, `R040`) moved cities on specific dates. We must join orders to restaurants where `order_ts >= effective_from` (using the latest version before the order timestamp) to prevent row multiplication.
- **Payments**: Contains charges (positive values) and refunds (negative values). Net Revenue must combine these transaction types correctly.
- **Order Status Events**: Used to track lifecycle (timestamps for `placed` and `delivered` to measure average delivery time).

In [6]:
# 1. Inspect payments - look for multiple payments per order (charges/refunds) or oddities
payments_summary = q("""
    SELECT
        COUNT(*) as total_rows,
        COUNT(DISTINCT payment_id) as unique_payments,
        COUNT(DISTINCT order_id) as unique_orders_with_payments,
        SUM(CASE WHEN amount < 0 THEN 1 ELSE 0 END) as refund_count,
        SUM(CASE WHEN amount >= 0 THEN 1 ELSE 0 END) as charge_count
    FROM payments
""")

print("--- PAYMENTS PROFILE ---")
display(payments_summary)

# Let's also inspect the duplicate restaurants anomaly
duplicate_restaurants = q("""
    SELECT restaurant_id, COUNT(*) as record_count
    FROM restaurants
    GROUP BY restaurant_id
    HAVING COUNT(*) > 1
""")

print("\n--- DUPLICATE RESTAURANTS ---")
display(duplicate_restaurants)

# Print a sample of the duplicate restaurant records to see why they exist
if not duplicate_restaurants.empty:
    dup_ids = tuple(duplicate_restaurants['restaurant_id'].tolist())
    display(q(f"SELECT * FROM restaurants WHERE restaurant_id IN {dup_ids} ORDER BY restaurant_id, effective_from"))

--- PAYMENTS PROFILE ---


,total_rows,unique_payments,unique_orders_with_payments,refund_count,charge_count
0,16423,16393,14494,1899.0,14524.0



--- DUPLICATE RESTAURANTS ---


,restaurant_id,record_count
0,R007,2
1,R021,2
2,R040,2


,restaurant_id,restaurant_name,cuisine,city,effective_from
0,R007,Royal Japanese 7,Japanese,Bristol,2025-01-01
1,R007,Royal Japanese 7,Japanese,London,2026-04-14
2,R021,Grand Thai 21,Thai,Manchester,2025-01-01
3,R021,Grand Thai 21,Thai,LEEDS,2026-05-06
4,R040,Corner Vegan 40,Vegan,London,2025-01-01
5,R040,Corner Vegan 40,Vegan,Bristol,2026-05-27


### 1a. Table Structure and Key Uniqueness Checks
Let's check if the primary keys (`order_id`, `event_id`, `payment_id`, `restaurant_id`, `customer_id`) are indeed unique in their respective source tables, and check for null values.

In [5]:
print("Checking Uniqueness of Keys and Null Values:\n")

# 1. Orders
print("--- ORDERS ---")
print("Total rows:", len(tables['orders']))
print("Unique order_ids:", tables['orders']['order_id'].nunique())
print("Null counts:\n", tables['orders'].isnull().sum(), "\n")

# 2. Customers
print("--- CUSTOMERS ---")
print("Total rows:", len(tables['customers']))
print("Unique customer_ids:", tables['customers']['customer_id'].nunique())
print("Null counts:\n", tables['customers'].isnull().sum(), "\n")

# 3. Restaurants
print("--- RESTAURANTS ---")
print("Total rows:", len(tables['restaurants']))
print("Unique restaurant_ids:", tables['restaurants']['restaurant_id'].nunique())
print("Null counts:\n", tables['restaurants'].isnull().sum(), "\n")

# 4. Order Status Events
print("--- ORDER STATUS EVENTS ---")
print("Total rows:", len(tables['order_status_events']))
print("Unique event_ids:", tables['order_status_events']['event_id'].nunique())
print("Unique statuses:", tables['order_status_events']['status'].unique())
print("Null counts:\n", tables['order_status_events'].isnull().sum(), "\n")

Checking Uniqueness of Keys and Null Values:

--- ORDERS ---
Total rows: 14494
Unique order_ids: 14494
Null counts:
 order_id         0
customer_id      0
restaurant_id    0
order_ts         0
delivery_fee     0
dtype: int64 

--- CUSTOMERS ---
Total rows: 4200
Unique customer_ids: 4200
Null counts:
 customer_id    0
signup_date    0
home_city      0
dtype: int64 

--- RESTAURANTS ---
Total rows: 67
Unique restaurant_ids: 64
Null counts:
 restaurant_id      0
restaurant_name    0
cuisine            0
city               0
effective_from     0
dtype: int64 

--- ORDER STATUS EVENTS ---
Total rows: 56812
Unique event_ids: 56812
Unique statuses: ['accepted' 'placed' 'delivered' 'picked_up' 'cancelled']
Null counts:
 event_id    0
order_id    0
status      0
event_ts    0
dtype: int64 



### 1b. Inspecting Order Lines and Payments
Let's check the grain of `order_lines` (which should be order + line item) and investigate how `payments` link to `orders` (e.g. can an order have multiple payments?).

## 2. Plan

*Before you write the main code, write your plan in your own words: what one row of your final table is, which tables you will join and in what order, how many rows you expect at each step, and how you will handle the problems you found.*

### 2. Implementation Plan

1. **Silver Layer (Clean Orders)**:
   - Clean the `restaurants` table by determining the active window for each version of a restaurant.
   - Filter out `cancelled` orders from our main metrics (or handle them separately, since delivery time is not applicable for cancelled orders).
   - Get `placed` and `delivered` timestamps from `order_status_events` to compute `delivery_time_minutes`.
   - Calculate net payment amount (sum of charges + refunds) per `order_id` from the `payments` table.
   - Join `orders` with the correct temporal version of `restaurants` to get the correct `city` and `cuisine`.

2. **Gold Layer (Reporting Table)**:
   - Group the clean orders by **Week** (starting Monday, derived from `order_ts`), **Cuisine**, and **City**.
   - Calculate `number_of_orders`, `net_revenue` (sum of payment amounts), and `avg_delivery_time` (average of delivery times).

## 3. Build

*Build as far as you can in the time. Short, re-runnable cells are better than one long cell.*

Add text here as you go along. Show that you are staying in control of the build!

### 2a. Building the Silver Layer (Clean Orders table)

We will write SQL to clean and enrich our `orders` data. This step will:
1. **Resolve Restaurant Slowly Changing Dimensions**: We map each order to the correct historical version of the restaurant where `order_ts >= effective_from` is satisfied for the most recent version.
2. **Aggregate Payments**: For each `order_id`, we compute the total net amount (charges + refunds).
3. **Compute Delivery Times**: Extract the `placed` and `delivered` timestamps from `order_status_events` to compute the actual delivery duration in minutes.
4. **Handle Cancelled Orders**: Flag cancelled orders so we can decide whether to include or exclude them from net revenue and order counts.

In [8]:
# 1. Create temporary tables/views to correctly join SCD restaurants
# For each restaurant version, determine its active window (effective_from to next change)
con.execute("""
CREATE OR REPLACE TEMP VIEW restaurants_clean AS
SELECT
    restaurant_id,
    restaurant_name,
    cuisine,
    city,
    effective_from,
    LEAD(effective_from) OVER (PARTITION BY restaurant_id ORDER BY effective_from ASC) as effective_to
FROM restaurants
""")

# 2. Pre-aggregate payments to get net amount per order
con.execute("""
CREATE OR REPLACE TEMP VIEW payments_clean AS
SELECT
    order_id,
    SUM(amount) as net_amount_paid
FROM payments
GROUP BY order_id
""")

# 3. Pivot status events to easily extract placed and delivered timestamps
con.execute("""
CREATE OR REPLACE TEMP VIEW order_timestamps AS
SELECT
    order_id,
    MIN(CASE WHEN status = 'placed' THEN CAST(event_ts AS TIMESTAMP) END) as placed_ts,
    MIN(CASE WHEN status = 'delivered' THEN CAST(event_ts AS TIMESTAMP) END) as delivered_ts,
    MAX(CASE WHEN status = 'cancelled' THEN 1 ELSE 0 END) as is_cancelled
FROM order_status_events
GROUP BY order_id
""")

# 4. Assemble the Silver Orders table
con.execute("""
CREATE OR REPLACE TABLE silver_orders AS
SELECT
    o.order_id,
    o.customer_id,
    o.restaurant_id,
    r.restaurant_name,
    r.cuisine,
    r.city,
    CAST(o.order_ts AS TIMESTAMP) as order_ts,
    t.placed_ts,
    t.delivered_ts,
    t.is_cancelled,
    EPOCH_MS(t.delivered_ts - t.placed_ts) / 1000 / 60.0 as delivery_time_minutes,
    COALESCE(p.net_amount_paid, 0.0) as net_amount_paid
FROM orders o
LEFT JOIN restaurants_clean r
  ON o.restaurant_id = r.restaurant_id
  AND CAST(o.order_ts AS TIMESTAMP) >= CAST(r.effective_from AS TIMESTAMP)
  AND (CAST(o.order_ts AS TIMESTAMP) < CAST(r.effective_to AS TIMESTAMP) OR r.effective_to IS NULL)
LEFT JOIN payments_clean p ON o.order_id = p.order_id
LEFT JOIN order_timestamps t ON o.order_id = t.order_id
""")

# Verify row counts and look for anomalies
print("Original orders size:", len(tables['orders']))
silver_count = q("SELECT COUNT(*) as count, COUNT(DISTINCT order_id) as unique_ids FROM silver_orders")
display(silver_count)
print("Sample of silver_orders:")
display(q("SELECT * FROM silver_orders LIMIT 5"))

Original orders size: 14494


,count,unique_ids
0,14494,14494


Sample of silver_orders:


,order_id,customer_id,restaurant_id,restaurant_name,cuisine,city,order_ts,placed_ts,delivered_ts,is_cancelled,delivery_time_minutes,net_amount_paid
0,O011178,C03419,R029,Royal Thai 29,Thai,Manchester,2026-06-02 17:02:06,2026-06-02 17:02:06,2026-06-02 17:42:31,0,40.416667,27.49
1,O005468,C03649,R047,Spice Japanese 47,Japanese,Birmingham,2026-04-15 13:18:28,2026-04-15 13:18:28,2026-04-15 13:43:36,0,25.133333,48.49
2,O013140,C00821,R058,Grand Indian 58,Indian,Manchester,2026-06-19 20:52:02,2026-06-19 20:52:02,2026-06-19 21:36:12,0,44.166667,43.49
3,O004726,C02991,R054,Grand Mexican 54,Mexican,Manchester,2026-04-09 16:12:06,2026-04-09 16:12:06,2026-04-09 17:05:15,0,53.150000,81.99
4,O011660,C02706,R060,Blue Burgers 60,Burgers,Leeds,2026-06-06 18:29:58,2026-06-06 18:29:58,2026-06-06 19:17:09,0,47.183333,42.49


## 4. Check

*Show how you know your answer is right. What did you compare, count or spot check, and what did the checks show? If a check failed, say what you changed.*

Add text here as you go along. Show that you are staying in control of the build!

## 5. Prompts and AI notes

*Paste each prompt you used, as you sent it. Then note what you accepted, what you changed or rejected, and why. Include at least one example where the AI was wrong or needed correcting, if that happened.*

Add prompts here.

## 6. What I could not verify

*Be honest about what remains unchecked, and what you would do next with more time.*

Add text here.

## Self-check before you stop

Tick these off in your head or in a text cell. They mirror the three things the real test is marked on.

**How was the code generated?**
- [ ] Every block of code that matters has a prompt or a source associated with it.
- [ ] I can say what each block is meant to do and what output I expected.

**Evidence of checking**
- [ ] For every table I built, I checked the number of rows against what I expected, and I can say why.
- [ ] I checked that keys I treat as unique are unique, and I looked for duplicates.
- [ ] My final table agrees with the source data in at least one total, and I show the comparison.
- [ ] I looked at individual records, not only at totals.
- [ ] I wrote down at least one thing the checks caught, and what I changed (if possible).

**Interpretation in my own words**
- [ ] I state each decision (week, which orders count, net revenue, delivery time) and why I chose it.
- [ ] I say what the figures can and cannot be trusted for.
- [ ] Section 6 lists what I did not verify, honestly.